# --------------------------------------Introduction--------------------------------------
## A categorical variable takes only a limited number of values.

* Consider a survey that asks how often we eat breakfast and provides four options: "Never", "Rarely", "Most days", or "Every day". In this case,         the data is categorical, because responses fall into a fixed set of categories.
* If people responded to a survey about which what brand of car they owned, the responses would fall into categories like "Honda", "Toyota", and          "Ford". In this case, the data is also categorical.

##### we will get an error if we try to plug these variables into most machine learning models in Python without preprocessing them first. In this tutorial, we'll compare three approaches that you can use to prepare your categorical data.

#### Remember the actual definition: categorical = a limited, fixed, repeating set of values that represent group membership.



### Corrected order:

* Split into numeric and categorical (your instinct here is right — RandomForest can't take raw strings).
* Impute missing values in categorical columns first — SimpleImputer(strategy='most_frequent') or strategy='constant', fill_value='Missing'.
* Then encode the (now null-free) categorical columns with Ordinal or One-Hot.
* Impute missing values in numeric columns (mean/median) — this can happen anytime before merging, order relative to step 2/3 doesn't matter since it's a separate column set.
* Merge numeric + encoded categorical back together.
* Feed into RandomForestRegressor.

In [120]:
#Import libraries
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [121]:
#Import melbourne data (csv)
data_path="melb_data.csv"
data=pd.read_csv(data_path)

In [122]:
#quickly inspecting data
data.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


In [123]:
# generate a statistical summary of the data
data.describe()

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.000000,1.358000e+04,13580.000000,13580.000000,13580.000000,13580.000000,13518.000000,13580.000000,7130.000000,8205.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,1.075684e+06,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,-37.809203,144.995216,7454.417378
std,0.955748,6.393107e+05,5.868725,90.676964,0.965921,0.691712,0.962634,3990.669241,541.014538,37.273762,0.079260,0.103916,4378.581772
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1196.000000,-38.182550,144.431810,249.000000
25%,2.000000,6.500000e+05,6.100000,3044.000000,2.000000,1.000000,1.000000,177.000000,93.000000,1940.000000,-37.856822,144.929600,4380.000000
50%,3.000000,9.030000e+05,9.200000,3084.000000,3.000000,1.000000,2.000000,440.000000,126.000000,1970.000000,-37.802355,145.000100,6555.000000
75%,3.000000,1.330000e+06,13.000000,3148.000000,3.000000,2.000000,2.000000,651.000000,174.000000,1999.000000,-37.756400,145.058305,10331.000000
max,10.000000,9.000000e+06,48.100000,3977.000000,20.000000,8.000000,10.000000,433014.000000,44515.000000,2018.000000,-37.408530,145.526350,21650.000000


In [124]:
# describes the memory layout, data type, and size of the elements
data.dtypes

Suburb            object
Address           object
Rooms              int64
Type              object
Price            float64
Method            object
SellerG           object
Date              object
Distance         float64
Postcode         float64
Bedroom2         float64
Bathroom         float64
Car              float64
Landsize         float64
BuildingArea     float64
YearBuilt        float64
CouncilArea       object
Lattitude        float64
Longtitude       float64
Regionname        object
Propertycount    float64
dtype: object

In [125]:
# Print columns
data.columns

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude',
       'Longtitude', 'Regionname', 'Propertycount'],
      dtype='object')

In [126]:
# Drop entire row for null target =>(Price)
data.dropna(subset=["Price"],axis=0,inplace=True)

# select target
y=data.Price

# select features 
X=data.drop(["Price"],axis=1)

# split data into two seperate parts tarin and test
train_X,valid_X,train_y,valid_y=train_test_split(X,y,train_size=0.8,test_size=0.2,random_state=0)

# Cardinality = the number of unique values in a column. A column like Sex (male/female) has cardinality 2
# Select categorical columns with relatively low cardinality (convenient but arbitrary) here=> less than 10
low_cardinality_cols=[]
for col in train_X.columns:
# The nunique() method in the Python Pandas library counts the number of unique (distinct) values in a dataset
    if train_X[col].nunique()<10 and train_X[col].dtype=="object": 
        low_cardinality_cols.append(col)
print(f"Low Cardinality columns: {low_cardinality_cols}")

# Select numerical columns
numerical_cols=[]
for col in train_X.columns:
    if train_X[col].dtype in ["int64","float64"]:
        numerical_cols.append(col)
print(f"Numerical columns: {numerical_cols}")

# Keep selected columns only
my_columns=low_cardinality_cols+numerical_cols
train_X_copy=train_X[my_columns].copy()
valid_X_copy=valid_X[my_columns].copy()

Low Cardinality columns: ['Type', 'Method', 'Regionname']
Numerical columns: ['Rooms', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude', 'Propertycount']


In [127]:
# Make a function to calculate Mean Absolute Error
def calculate_error(train_X,valid_X,train_y,valid_y):
    model=RandomForestRegressor(n_estimators=100,random_state=0) # n_estimators means the number of trees in a model
    model.fit(train_X,train_y)
    predection=model.predict(valid_X)
    error=mean_absolute_error(valid_y,predection)
    return error

In [128]:
# Get the list of categorical variables
categorical_variables=[]
for col in train_X_copy.columns:
    if train_X_copy[col].dtype=="object":
        categorical_variables.append(col)
print(f"Categorical Variables: {categorical_variables}")

Categorical Variables: ['Type', 'Method', 'Regionname']


## --------------------------------------------Three Approaches--------------------------------------------

#### Approach 1 (Drop Categorical Variables):

* The easiest approach to dealing with categorical variables is to simply remove them from the dataset. This approach will only work well if the columns did not contain useful information.

Note: There are two parts of our data (Numerical and Categorical). And there are missing values in dataset, so we will manipulate both parts using (Imputation and Imputation with extension because droping columns gives wrost result.).

=>For Approach 1 (two seperate step to calculate Mean absolute error)
1. (imputation for numeric + drop categorical because imputation for this approach is valueless)
2. (imputation with extension for numeric + drop categorical because imputation with extension for this approach is valueless)

In [129]:
# Impute Numeric columns and Drop Categorical Variables

# Make a copy 
impute_train_X_copy=train_X_copy.copy()
impute_valid_X_copy=valid_X_copy.copy()

# Impute numeric columns
numeric_imputer=SimpleImputer()

impute_train_X_copy[numerical_cols]=numeric_imputer.fit_transform(impute_train_X_copy[numerical_cols])
impute_valid_X_copy[numerical_cols]=numeric_imputer.transform(impute_valid_X_copy[numerical_cols])
# SimpleImputer on specific columns like this already keeps it as a DataFrame. No conversion needed.
# Column names are already preserved when we impute specific columns by name like this anyway.
# So we do not need to make DataFrame and copy columns

# Drop columns for Categorical variables
droped_impute_train_X_copy=impute_train_X_copy.drop(categorical_variables,axis=1)
droped_impute_valid_X_copy=impute_valid_X_copy.drop(categorical_variables,axis=1)

# Print Mean Absolute Error
print("Approach 1 (Impute Numeric columns and Drop Categorical Variables)")
print(f"Mean Absolute Error: {calculate_error(droped_impute_train_X_copy,droped_impute_valid_X_copy,train_y,valid_y)}")



# Impute Numeric columns with extension and Drop Categorical Variables
numeric_imputer1=SimpleImputer()
# Make a copy 
impute_extension_train_X_copy=train_X_copy.copy()
impute_extension_valid_X_copy=valid_X_copy.copy()

# Missing values in numeric columns
missing_values_numeric_columns=[]
for col in numerical_cols:
    if impute_extension_train_X_copy[col].isnull().any():
        missing_values_numeric_columns.append(col)

# Impute numeric columns with extension
for  col in missing_values_numeric_columns:
    impute_extension_train_X_copy[col+" was missing"]=impute_extension_train_X_copy[col].isnull()
    impute_extension_valid_X_copy[col+" was missing"]=impute_extension_valid_X_copy[col].isnull()

impute_extension_train_X_copy[numerical_cols]=numeric_imputer1.fit_transform(impute_extension_train_X_copy[numerical_cols])
impute_extension_valid_X_copy[numerical_cols]=numeric_imputer1.transform(impute_extension_valid_X_copy[numerical_cols])
# SimpleImputer on specific columns like this already keeps it as a DataFrame. No conversion needed.
# Column names are already preserved when we impute specific columns by name like this anyway.
# So we do not need to make DataFrame and copy columns

# Drop columns for Categorical variables
droped_impute_extension_train_X_copy=impute_extension_train_X_copy.drop(categorical_variables,axis=1)
droped_impute_extension_valid_X_copy=impute_extension_valid_X_copy.drop(categorical_variables,axis=1)

# Print Mean Absolute Error
print("Approach 1 (Impute Numeric columns with extension and Drop Categorical Variables)")
print(f"Mean Absolute Error: {calculate_error(droped_impute_extension_train_X_copy,droped_impute_extension_valid_X_copy,train_y,valid_y)}")

Approach 1 (Impute Numeric columns and Drop Categorical Variables)
Mean Absolute Error: 169237.0268668034
Approach 1 (Impute Numeric columns with extension and Drop Categorical Variables)
Mean Absolute Error: 169795.45249719475
